# NDT7 (M-Lab) Data Prep — Vietnam Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/vn/mlab_vn_clean.parquet` (24.7M raw NDT7 test records, already ISP-classified and province-joined via
per-IP lookup + point-in-polygon) into province x quarter format, split into Broadband and
Mobile/Cellular parts, mirroring the same structure across all three NDT7 "tigger" countries
(Cambodia/Thailand/Vietnam).

**Rebuilt to use DuckDB instead of a manual pyarrow-batch-streaming loop** — DuckDB reads the
parquet file directly and does the tile-binning + GROUP BY aggregation out-of-core (no manual
batching code needed, no risk of the memory issues the streaming version was written to avoid).
The tile-binning and weighted-aggregation formulas are byte-for-byte unchanged from the pandas
version — verified against the prior pandas-based export (float-precision-only differences,
~1e-13, from AVG() accumulation order).

No province-name mapping needed — Vietnam's raw `province` values already match `vietnam_reference.csv`.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100 & n_tiles >= 5`.

**Outputs:**
- `data/exports/ndt7_vietnam_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_vietnam_province_quarterly.csv` — Mobile/Cellular
  (renamed from `ndt7_vietnam_mobile_...` to match Ookla's `ookla_mobile_<country>_...`
  naming convention — position of "mobile" now matches across both pipelines)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/vn/mlab_vn_clean.parquet'
VN_REF_CSV = '../../../data/reference/vietnam_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        LEAST(min_rtt, 2000) AS min_rtt,
        latitude, longitude, type, network_type, province,
        date_part('year', date) AS yr,
        date_part('quarter', date) AS qtr
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND latitude IS NOT NULL AND longitude IS NOT NULL
      AND province IS NOT NULL AND date IS NOT NULL
),
tiled AS (
    SELECT
        *,
        (CAST(yr AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
        CAST(FLOOR((longitude + 180) / 360 * {N_TILES}) AS BIGINT) AS tile_x_raw,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))) + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))) ) / pi()) / 2 * {N_TILES}) AS BIGINT) AS tile_y_raw
    FROM filtered
),
clipped AS (
    SELECT *,
        LEAST(GREATEST(tile_x_raw, 0), {N_TILES}-1) AS tile_x,
        LEAST(GREATEST(tile_y_raw, 0), {N_TILES}-1) AS tile_y
    FROM tiled
),
tile_id_cte AS (
    SELECT *, (CAST(tile_x AS VARCHAR) || '_' || CAST(tile_y AS VARCHAR)) AS tile_id
    FROM clipped
),
tile_agg AS (
    SELECT
        year_q, tile_id, type, network_type,
        AVG(mean_throughput_mbps) AS tile_mean,
        AVG(min_rtt) AS tile_lat,
        COUNT(*) AS test_count,
        mode(province) AS province
    FROM tile_id_cte
    GROUP BY year_q, tile_id, type, network_type
    HAVING COUNT(*) >= {MIN_TILE_TESTS}
)
SELECT * FROM tile_agg
"""

con = duckdb.connect()
tile_agg_all = con.execute(sql).df()
print(f"Tile x quarter x type x network rows (>= {MIN_TILE_TESTS} tests/tile): {len(tile_agg_all):,}")
print(f"Quarters covered: {sorted(tile_agg_all['year_q'].unique())}")
print(tile_agg_all['network_type'].value_counts())

Tile x quarter x type x network rows (>= 3 tests/tile): 7,062
Quarters covered: ['2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
network_type
broadband    5568
cellular     1057
hosting       437
Name: count, dtype: int64


### 3. Province-Level Weighted Aggregation (per network type)

In [3]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
        'total_tests': g['test_count'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [4]:
ref = pd.read_csv(VN_REF_CSV)

---
## Part 1 — Broadband

In [5]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 5,568


[broadband] province x quarter rows: 733 | reliable: 210 (28.6%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,An Giang,29.349034,112.389486,759.0,10.0,31.547373,2023,1,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,2023-Q1,Bà Rịa–Vũng Tàu,35.333411,112.670980,1558.0,7.0,28.657948,2023,1,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,2023-Q1,Bình Dương,31.278105,107.479716,747.0,11.0,25.175450,2023,1,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61
3,2023-Q1,Bình Phước,29.091671,124.483435,278.0,8.0,24.916346,2023,1,True,Southeast,2,1313000,3606.56,145,10074.62,322186.39
4,2023-Q1,Bình Thuận,31.337100,97.125557,314.0,5.0,29.936516,2023,1,True,South Central Coast,2,1498000,3090.17,155,8632.13,276055.50


In [6]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_vietnam_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 733 rows -> ../../../data/exports/ndt7_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,An Giang,2023-Q1,2023,1,29.349034,31.547373,112.389486,759.0,10.0,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,Bà Rịa–Vũng Tàu,2023-Q1,2023,1,35.333411,28.657948,112.670980,1558.0,7.0,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,Bình Dương,2023-Q1,2023,1,31.278105,25.175450,107.479716,747.0,11.0,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61


---
## Part 2 — Mobile/Cellular

In [7]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 1,057


[cellular] province x quarter rows: 396 | reliable: 4 (1.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bà Rịa–Vũng Tàu,22.548900,197.897375,16.0,2.0,14.552571,2023,1,False,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
1,2023-Q1,Bình Dương,9.071973,146.487286,7.0,2.0,3.658284,2023,1,False,Southeast,2,2564000,3663.54,901,10233.79,327276.61
2,2023-Q1,Bình Phước,38.013270,141.964667,3.0,1.0,NaN,2023,1,False,Southeast,2,1313000,3606.56,145,10074.62,322186.39
3,2023-Q1,Bình Thuận,40.451301,115.547764,89.0,3.0,31.502123,2023,1,False,South Central Coast,2,1498000,3090.17,155,8632.13,276055.50
4,2023-Q1,Bình Định,11.630636,278.179689,103.0,2.0,8.800083,2023,1,False,South Central Coast,2,1679000,3089.10,245,8629.14,275959.91


In [8]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_vietnam_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 396 rows -> ../../../data/exports/ndt7_mobile_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bà Rịa–Vũng Tàu,2023-Q1,2023,1,22.548900,14.552571,197.897375,16.0,2.0,False,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
1,Bình Dương,2023-Q1,2023,1,9.071973,3.658284,146.487286,7.0,2.0,False,Southeast,2,2564000,3663.54,901,10233.79,327276.61
2,Bình Phước,2023-Q1,2023,1,38.013270,NaN,141.964667,3.0,1.0,False,Southeast,2,1313000,3606.56,145,10074.62,322186.39


## Summary

- Input: Vietnam NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)